In [ ]:
import numpy as np
from tqdm import tqdm
import shutil

In [ ]:
!pip install Pillow

In [ ]:
# ------------- Number 1 -------------

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls /content/drive/MyDrive/

In [ ]:
# ------------- Number 2 -------------

!unzip /content/drive/MyDrive/Butterfly.zip -d Butterfly


In [ ]:
# ------------- Number 3 -------------

import os
print(os.listdir("Butterfly/Butterfly"))

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

folder_path = "Butterfly/Butterfly/train"

files = sorted(os.listdir(folder_path))

for fname in files:
    img_path = os.path.join(folder_path, fname)

    # Skip non-image files (optional)
    if not fname.lower().endswith((".jpg")):
        continue

    img = Image.open(img_path)

    plt.imshow(img)
    plt.axis("off")
    plt.show()

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

folder_path = "Butterfly/Butterfly/test"

files = sorted(os.listdir(folder_path))

for fname in files:
    img_path = os.path.join(folder_path, fname)

    if not fname.lower().endswith((".jpg")):
        continue

    img = Image.open(img_path)

    plt.imshow(img)
    plt.axis("off")
    plt.show()

In [ ]:
import pandas as pd

df = pd.read_csv("Butterfly/Butterfly/Training_set.csv")
df.head()

In [ ]:
import pandas as pd

df = pd.read_csv("Butterfly/Butterfly/Testing_set.csv")
df.head()

In [ ]:
df = pd.read_csv("Butterfly/Butterfly/Training_set.csv")

missing_labels = df[df['label'].isna() | (df['label'] == "")]
print("Missing labels:")
print(missing_labels)

In [ ]:
folder_path = "Butterfly/Butterfly/train"
column_name = "filename"

bad_images = []

for img_name in df[column_name]:

    img_path = os.path.join(folder_path, img_name)

    if not os.path.exists(img_path):
        print(f"Missing image: {img_path}")
        bad_images.append(img_path)
        continue

    try:
        img = Image.open(img_path)
        img.verify()
    except Exception as e:
        print(f"Corrupted header: {img_path} - Error: {e}")
        bad_images.append(img_path)
        continue

    try:
        img = Image.open(img_path)
        img.load()
    except Exception as e:
        print(f"Image fails to load fully: {img_path} - Error: {e}")
        bad_images.append(img_path)

print("\nTotal bad or missing images:", len(bad_images))


In [ ]:
from PIL import Image

folder = "Butterfly/Butterfly/train"
output_folder = "Butterfly_resized"

os.makedirs(output_folder, exist_ok=True)

for file in os.listdir(folder):
    if file.lower().endswith((".jpg", ".jpeg", ".png")):
        path = os.path.join(folder, file)
        img = Image.open(path)

        img_resized = img.resize((96, 96))

        img_resized.save(os.path.join(output_folder, file))

print("All images resized!")

In [ ]:
# ------------- Number 4 -------------

import os
from PIL import Image

folder = "Butterfly/Butterfly/test"
output_folder = "Butterfly_resized_test"

os.makedirs(output_folder, exist_ok=True)

count = 0

for file in os.listdir(folder):
    if file.lower().endswith((".jpg", ".jpeg", ".png")):
        count += 1
        path = os.path.join(folder, file)
        img = Image.open(path)

        img_resized = img.resize((96, 96))
        img_resized.save(os.path.join(output_folder, file))

print(f"Total images processed: {count}")
print("All images resized!")

In [ ]:
folder = "Butterfly_resized"
target_size = (96, 96)

wrong_images = []

for file in os.listdir(folder):
    if file.lower().endswith((".jpg", ".jpeg", ".png")):
        path = os.path.join(folder, file)

        try:
            img = Image.open(path)
            if img.size != target_size:
                wrong_images.append((file, img.size))
        except:
            print("Cannot open:", file)

print("\nChecked images:", len(os.listdir(folder)))
print("Images with WRONG size:", len(wrong_images))

for f, size in wrong_images:
    print(f"❌ {f} has size {size}, expected {target_size}")

if len(wrong_images) == 0:
    print("\n✅ All images are correctly resized!")


In [ ]:
# ------------- Number 5 -------------


from PIL import Image
import os
import shutil
from google.colab import files

input_folder = "Butterfly/Butterfly/test"
output_folder = "Butterfly_resized_test"
target_size = (96, 96)

if os.path.exists(output_folder):
    shutil.rmtree(output_folder)
os.makedirs(output_folder, exist_ok=True)

image_files = [f for f in os.listdir(input_folder) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

for file in image_files:
    input_path = os.path.join(input_folder, file)
    output_path = os.path.join(output_folder, file)
    try:
        img = Image.open(input_path)
        img_resized = img.resize(target_size)
        img_resized.save(output_path)
    except Exception as e:
        print(f"Cannot process {file} - {e}")

wrong_images = []
resized_files = [f for f in os.listdir(output_folder) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

for file in resized_files:
    path = os.path.join(output_folder, file)
    try:
        img = Image.open(path)
        if img.size != target_size:
            wrong_images.append((file, img.size))
    except Exception as e:
        print(f"Cannot open {file} - {e}")

shutil.make_archive("Butterfly_resized_test", 'zip', "Butterfly_resized_test")
files.download("Butterfly_resized_test.zip")


In [ ]:
folder = "Butterfly_resized"
duplicate_folder = "duplicate_photos"
unique_folder = "unique_photos"
threshold = 0.001

os.makedirs(duplicate_folder, exist_ok=True)
os.makedirs(unique_folder, exist_ok=True)

def load_image(path):
    img = Image.open(path)
    return np.array(img).astype("float32") / 255.0


def mse(img1, img2):
    return np.mean((img1 - img2) ** 2)

files = sorted([
    f for f in os.listdir(folder)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

duplicates = []
deleted_files = set()

print("Comparing images...\n")

for i in tqdm(range(len(files)), desc="Outer Loop"):
    img1_name = files[i]
    img1_path = os.path.join(folder, img1_name)

    if img1_name in deleted_files:
        continue

    img1 = load_image(img1_path)

    for j in range(i + 1, len(files)):
        img2_name = files[j]
        img2_path = os.path.join(folder, img2_name)

        if img2_name in deleted_files:
            continue

        img2 = load_image(img2_path)
        score = mse(img1, img2)

        if score < threshold:
            duplicates.append([img1_name, img2_name, score])

            shutil.copy(img2_path, os.path.join(duplicate_folder, img2_name))

            os.remove(img2_path)
            deleted_files.add(img2_name)

print("\nSaving unique images...")

unique_images = []

for img_name in files:
    if img_name not in deleted_files:
        src_path = os.path.join(folder, img_name)
        dest_path = os.path.join(unique_folder, img_name)
        shutil.copy(src_path, dest_path)
        unique_images.append(img_name)

pd.DataFrame(unique_images, columns=["unique_image"]).to_csv("unique_images.csv", index=False)

df = pd.DataFrame(duplicates, columns=["image_1", "image_2", "mse_score"])
df.to_csv("duplicate_pairs.csv", index=False)

print("\nCSV saved as: duplicate_pairs.csv")
print("Unique images CSV saved as: unique_images.csv")
print(f"Total duplicates found: {len(duplicates)}")
print(f"Unique images saved to: {unique_folder}")
print(f"Duplicates saved to: {duplicate_folder}")

for img1_name, img2_name, score in duplicates:
    path1 = os.path.join(folder, img1_name)
    path2 = os.path.join(duplicate_folder, img2_name)

    if not os.path.exists(path1):
        continue

    img1 = Image.open(path1)
    img2 = Image.open(path2)

    plt.figure(figsize=(8, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(img1)
    plt.title(img1_name)
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(img2)
    plt.title(f"{img2_name}\nMSE={score:.6f}")
    plt.axis("off")

    plt.show()

from google.colab import files

print("\nPreparing files for download...")

files.download("duplicate_pairs.csv")
files.download("unique_images.csv")

!zip -r duplicate_photos.zip duplicate_photos
!zip -r unique_photos.zip unique_photos

files.download("duplicate_photos.zip")
files.download("unique_photos.zip")

In [ ]:
# ------------- Number 6 -------------

!ls /content/drive/MyDrive/Butterfly_Google_Colab/

In [ ]:
# ------------- Number 7 -------------


!unzip /content/drive/MyDrive/Butterfly_Google_Colab/unique_photos.zip -d unique_photos

In [ ]:
import pandas as pd
from google.colab import files

unique_df = pd.read_csv("/content/drive/MyDrive/Butterfly_Google_Colab/unique_images.csv")
train_df = pd.read_csv("Butterfly/Butterfly/Training_set.csv")

unique_df["unique_image"] = unique_df["unique_image"].astype(str).str.strip()
train_df["filename"] = train_df["filename"].astype(str).str.strip()

if not train_df["filename"].str.contains(".jpg").any():
    train_df["filename_with_ext"] = train_df["filename"] + ".jpg"
else:
    train_df["filename_with_ext"] = train_df["filename"]

merged_df = unique_df.merge(
    train_df,
    left_on="unique_image",
    right_on="filename_with_ext",
    how="left"
)


final_df = merged_df[["unique_image", "label"]]

output_name = "unique_images_with_labels.csv"
final_df.to_csv(output_name, index=False)

files.download(output_name)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("/content/drive/MyDrive/Butterfly_Google_Colab/unique_images_with_labels.csv")

label_counts = df["label"].value_counts()

num_classes = df['label'].nunique()

plt.figure(figsize=(20, 10))
plt.bar(label_counts.index, label_counts.values)
plt.xlabel("Class Labels")
plt.ylabel("Number of Images")
plt.title(f"Class Composition of Unique Images (Total classes: {num_classes})")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
import os
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

image_folder = "unique_photos/unique_photos"

hist_r = np.zeros(256, dtype=np.int64)
hist_g = np.zeros(256, dtype=np.int64)
hist_b = np.zeros(256, dtype=np.int64)

image_files = [f for f in os.listdir(image_folder) if f.endswith((".jpg", ".png"))]


for filename in tqdm(image_files, desc="Processing images"):
    img_path = os.path.join(image_folder, filename)
    img = Image.open(img_path).convert("RGB")
    img_array = np.array(img)

    hist_r += np.bincount(img_array[:, :, 0].flatten(), minlength=256)
    hist_g += np.bincount(img_array[:, :, 1].flatten(), minlength=256)
    hist_b += np.bincount(img_array[:, :, 2].flatten(), minlength=256)

plt.figure(figsize=(15,5))
plt.bar(range(256), hist_r, color='red', alpha=0.5, label='Red channel')
plt.bar(range(256), hist_g, color='green', alpha=0.5, label='Green channel')
plt.bar(range(256), hist_b, color='blue', alpha=0.5, label='Blue channel')
plt.title("Color Distribution Across All Images")
plt.xlabel("Pixel Intensity")
plt.ylabel("Frequency")
plt.legend()
plt.show()

In [ ]:
import os
from PIL import Image
from tqdm import tqdm

image_folder = "unique_photos/unique_photos"

num_grayscale = 0
num_color = 0

image_files = [f for f in os.listdir(image_folder) if f.endswith((".jpg", ".png"))]

for filename in tqdm(image_files, desc="Checking images"):
    img_path = os.path.join(image_folder, filename)
    img = Image.open(img_path)

    if img.mode == 'L':
        num_grayscale += 1
    elif img.mode == 'RGB':
        num_color += 1

total_images = num_grayscale + num_color
print(f"Grayscale images: {num_grayscale} ({num_grayscale/total_images*100:.2f}%)")
print(f"Color images: {num_color} ({num_color/total_images*100:.2f}%)")


In [ ]:
# ------------- Number 8 -------------


import os
import zipfile
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

csv_file = "/content/drive/MyDrive/Butterfly_Google_Colab/unique_images_with_labels.csv"
df = pd.read_csv(csv_file)

zip_path = "/content/drive/MyDrive/Butterfly_Google_Colab/unique_photos.zip"
extract_path = "unique_photos/unique_photos"

with zipfile.ZipFile(zip_path, 'r') as z:
    for member in tqdm(z.namelist(), desc="Extracting ZIP"):
        z.extract(member, extract_path)

num_classes = df.iloc[:,1].nunique()
print("Number of classes:", num_classes)

train_df, test_df = train_test_split(df, test_size=0.2, shuffle=True, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.1, shuffle=True, random_state=42)

train_transforms = T.Compose([
    T.Resize((96,96)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.RandomResizedCrop(224),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_test_transforms = T.Compose([
    T.Resize((96,96)),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

class CustomDataset(Dataset):
    def __init__(self, df, folder, transforms):
        self.paths = df.iloc[:,0].values
        self.labels = df.iloc[:,1].values
        self.folder = folder
        self.transforms = transforms

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img_path = os.path.join(self.folder, self.paths[idx])
        img = Image.open(img_path).convert("RGB")
        img = self.transforms(img)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, label

train_dataset = CustomDataset(train_df, extract_path, train_transforms)
val_dataset = CustomDataset(val_df, extract_path, val_test_transforms)
test_dataset = CustomDataset(test_df, extract_path, val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class SimpleCNN(nn.Module):
    def __init__(self, input_channels=3, num_classes=10, lr=1e-3):
        super(SimpleCNN, self).__init__()

        self.conv1 = nn.Conv2d(input_channels, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)


        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)


        self.dropout = nn.Dropout(0.5)


        self.fc = nn.Linear(128 * 12 * 12, num_classes)


        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(self.parameters(), lr=lr)

    def forward(self, x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = nn.ReLU()(x)
        x = self.pool(x)


        x = self.conv2(x)
        x = self.bn2(x)
        x = nn.ReLU()(x)
        x = self.pool(x)


        x = self.conv3(x)
        x = nn.ReLU()(x)
        x = self.pool(x)


        x = x.view(x.size(0), -1)
        x = self.dropout(x)


        x = self.fc(x)
        return x


model = SimpleCNN(input_channels=3, num_classes=10, lr=1e-3)
print(model)


x = torch.randn(5, 3, 96, 96)
outputs = model(x)
print("Output shape:", outputs.shape)


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import pandas as pd
import torchvision.transforms as T
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns


class ButterflyDataset(Dataset):
    def __init__(self, csv_file, img_folder, transform=None):
        self.data = pd.read_csv(csv_file)
        self.img_folder = img_folder
        self.transform = transform

        self.classes = sorted(self.data['label'].unique())
        self.class_to_idx = {label: idx for idx, label in enumerate(self.classes)}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        filename = str(self.data.iloc[idx, 0])

        possible_files = [f for f in os.listdir(self.img_folder) if f.startswith(filename)]
        if len(possible_files) == 0:
            raise FileNotFoundError(f"No image found for: {filename}")

        img_path = os.path.join(self.img_folder, possible_files[0])
        image = Image.open(img_path).convert('RGB')

        label_str = self.data.iloc[idx, 1]
        label = self.class_to_idx[label_str]

        if self.transform:
            image = self.transform(image)

        return image, label



transform = T.Compose([
    T.Resize((96, 96)),
    T.ToTensor(),
    T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])




train_csv = "/content/drive/MyDrive/Butterfly_Google_Colab/unique_images_with_labels.csv"
train_folder = "unique_photos/unique_photos"

full_dataset = ButterflyDataset(train_csv, train_folder, transform)

val_size = int(0.2 * len(full_dataset))
train_size = len(full_dataset) - val_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)



class SimpleCNN(nn.Module):
    def __init__(self, input_channels=3, num_classes=10, lr=1e-3):
        super(SimpleCNN, self).__init__()

        self.conv1 = nn.Conv2d(input_channels, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.5)

        self.fc = nn.Linear(128 * 12 * 12, num_classes)

        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(self.parameters(), lr=lr)

    def forward(self, x):
        x = self.pool(nn.ReLU()(self.bn1(self.conv1(x))))
        x = self.pool(nn.ReLU()(self.bn2(self.conv2(x))))
        x = self.pool(nn.ReLU()(self.conv3(x)))

        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc(x)
        return x


    def train_step(self, features, labels):
        self.optimizer.zero_grad()
        outputs = self.forward(features)
        loss = self.criterion(outputs, labels)
        loss.backward()
        self.optimizer.step()
        return loss.item()


    def train_model(self, train_loader, val_loader, num_epochs=10, device='cpu'):
        train_losses = []
        val_losses = []

        for epoch in range(num_epochs):
            self.train()
            batch_losses = []

            for features, labels in train_loader:
                features, labels = features.to(device), labels.to(device)
                loss = self.train_step(features, labels)
                batch_losses.append(loss)

            avg_train = sum(batch_losses) / len(batch_losses)
            train_losses.append(avg_train)


            self.eval()
            val_loss_total = 0
            with torch.no_grad():
                for features, labels in val_loader:
                    features, labels = features.to(device), labels.to(device)
                    outputs = self.forward(features)
                    loss = self.criterion(outputs, labels)
                    val_loss_total += loss.item() * features.size(0)

            avg_val = val_loss_total / len(val_loader.dataset)
            val_losses.append(avg_val)

            print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")

        return train_losses, val_losses


    def evaluate(self, dataloader, device='cpu'):
        self.eval()
        preds = []
        labels_list = []
        total_loss = 0

        with torch.no_grad():
            for features, labels in dataloader:
                features, labels = features.to(device), labels.to(device)

                outputs = self.forward(features)
                loss = self.criterion(outputs, labels)
                total_loss += loss.item() * features.size(0)

                pred = torch.argmax(outputs, dim=1)
                preds.append(pred.cpu())
                labels_list.append(labels.cpu())

        preds = torch.cat(preds)
        labels_list = torch.cat(labels_list)
        avg_loss = total_loss / len(dataloader.dataset)

        return preds, labels_list, avg_loss


    def predict_image(self, img_path, transform, class_names, device='cpu'):
        image = Image.open(img_path).convert('RGB')
        image = transform(image).unsqueeze(0).to(device)

        self.eval()
        with torch.no_grad():
            output = self.forward(image)
            pred = torch.argmax(output, dim=1).item()

        return class_names[pred]


    def plot_losses(self, train, val):
        plt.figure(figsize=(8,5))
        plt.plot(train, label="Train Loss")
        plt.plot(val, label="Validation Loss")
        plt.legend()
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title("Training vs Validation Loss")
        plt.show()


    def plot_confusion_matrix(self, true_labels, predictions, class_names):
        cm = confusion_matrix(true_labels, predictions)
        plt.figure(figsize=(10,7))
        sns.heatmap(
            cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names
        )
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.title("Confusion Matrix")
        plt.show()


        fp = (cm.sum(axis=0) - np.diag(cm)).sum()
        fn = (cm.sum(axis=1) - np.diag(cm)).sum()
        print("\n--- Error Reflection ---")
        print(f"False Positives: {fp}")
        print(f"False Negatives: {fn}")
        if fp > fn:
            print("Model makes more FALSE POSITIVES.")
        elif fn > fp:
            print("Model makes more FALSE NEGATIVES.")
        else:
            print("Model makes similar FP and FN.")


    def save_model(self, path="model.pth"):
        torch.save(self.state_dict(), path)
        print(f"Model saved to {path}")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(full_dataset.classes)

model = SimpleCNN(num_classes=num_classes).to(device)

train_losses, val_losses = model.train_model(
    train_loader, val_loader, num_epochs=5, device=device
)

model.plot_losses(train_losses, val_losses)


preds, labels, val_loss = model.evaluate(val_loader, device=device)

print("\nAccuracy:", accuracy_score(labels, preds))
print("Precision:", precision_score(labels, preds, average="macro"))
print("Recall:", recall_score(labels, preds, average="macro"))
print("F1 Score:", f1_score(labels, preds, average="macro"))

model.plot_confusion_matrix(labels, preds, class_names=full_dataset.classes)

model.save_model("butterfly_model.pth")


In [ ]:
# import os
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import Dataset, DataLoader, random_split
# from PIL import Image
# import pandas as pd
# import torchvision.transforms as T
# import numpy as np
# import matplotlib.pyplot as plt
# from sklearn.metrics import confusion_matrix, f1_score
# import seaborn as sns
# import cv2
# from tqdm import tqdm


# class ButterflyDataset(Dataset):
#     def __init__(self, csv_file, img_folder, transform=None):
#         self.data = pd.read_csv(csv_file)
#         self.img_folder = img_folder
#         self.transform = transform
#         self.classes = sorted(self.data['label'].unique())
#         self.class_to_idx = {label: idx for idx, label in enumerate(self.classes)}

#     def __len__(self):
#         return len(self.data)

#     def __getitem__(self, idx):
#         filename = str(self.data.iloc[idx, 0])
#         img_file = [f for f in os.listdir(self.img_folder) if f.startswith(filename)][0]
#         image = Image.open(os.path.join(self.img_folder, img_file)).convert("RGB")
#         label = self.class_to_idx[self.data.iloc[idx, 1]]
#         if self.transform:
#             image = self.transform(image)
#         return image, label


# transform = T.Compose([
#     T.Resize((96, 96)),
#     T.ToTensor(),
#     T.Normalize([0.5]*3, [0.5]*3)
# ])

# train_csv = "/content/drive/MyDrive/Butterfly_Google_Colab/unique_images_with_labels.csv"
# train_folder = "unique_photos/unique_photos"

# dataset = ButterflyDataset(train_csv, train_folder, transform)

# val_size = int(0.2 * len(dataset))
# train_ds, val_ds = random_split(dataset, [len(dataset)-val_size, val_size])

# train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
# val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)


# def get_activation(name):
#     if name == "relu":
#         return nn.ReLU()
#     if name == "gelu":
#         return nn.GELU()
#     if name == "leakyrelu":
#         return nn.LeakyReLU(0.1)
#     raise ValueError("Invalid activation")


# class SimpleCNN(nn.Module):
#     def __init__(self, num_classes, activation):
#         super().__init__()
#         act = get_activation(activation)

#         self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
#         self.bn1 = nn.BatchNorm2d(32)

#         self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
#         self.bn2 = nn.BatchNorm2d(64)

#         self.conv3 = nn.Conv2d(64, 128, 3, padding=1)

#         self.pool = nn.MaxPool2d(2)
#         self.act = act
#         self.dropout = nn.Dropout(0.3)

#         self.fc = nn.Linear(128 * 12 * 12, num_classes)

#         self.criterion = nn.CrossEntropyLoss()
#         self.optimizer = optim.Adam(self.parameters(), lr=3e-4)

#     def forward(self, x):
#         x = self.pool(self.act(self.bn1(self.conv1(x))))
#         x = self.pool(self.act(self.bn2(self.conv2(x))))
#         x = self.pool(self.act(self.conv3(x)))
#         x = x.view(x.size(0), -1)
#         return self.fc(self.dropout(x))

#     def train_model(self, train_loader, val_loader, epochs, device, name):
#         print(f"\n Training {name}")
#         for epoch in range(epochs):
#             self.train()
#             running_loss = 0.0

#             pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
#             for x, y in pbar:
#                 x, y = x.to(device), y.to(device)

#                 self.optimizer.zero_grad()
#                 loss = self.criterion(self(x), y)
#                 loss.backward()
#                 self.optimizer.step()

#                 running_loss += loss.item()
#                 pbar.set_postfix(loss=loss.item())

#             avg_loss = running_loss / len(train_loader)
#             preds, labels = self.evaluate(val_loader, device)
#             f1 = f1_score(labels, preds, average="macro")

#             print(f"Epoch [{epoch+1}/{epochs}] | Loss: {avg_loss:.4f} | Val F1: {f1:.4f}")

#     def evaluate(self, loader, device):
#         self.eval()
#         preds, labels = [], []
#         with torch.no_grad():
#             for x, y in loader:
#                 out = self(x.to(device))
#                 preds.append(out.argmax(1).cpu())
#                 labels.append(y)
#         return torch.cat(preds), torch.cat(labels)

# class DeepCNN(SimpleCNN):
#     def __init__(self, num_classes, activation):
#         super().__init__(num_classes, activation)
#         act = get_activation(activation)

#         self.features = nn.Sequential(
#             nn.Conv2d(3, 32, 3, padding=1), act, nn.BatchNorm2d(32),
#             nn.Conv2d(32, 64, 3, padding=1), act, nn.BatchNorm2d(64),
#             nn.MaxPool2d(2),

#             nn.Conv2d(64, 128, 3, padding=1), act, nn.BatchNorm2d(128),
#             nn.Conv2d(128, 256, 3, padding=1), act, nn.BatchNorm2d(256),
#             nn.MaxPool2d(2),

#             nn.Conv2d(256, 256, 3, padding=1), act,
#             nn.MaxPool2d(2)
#         )

#         self.fc = nn.Linear(256 * 12 * 12, num_classes)
#         self.dropout = nn.Dropout(0.3)

#         self.criterion = nn.CrossEntropyLoss()
#         self.optimizer = optim.Adam(self.parameters(), lr=3e-4)

#     def forward(self, x):
#         x = self.features(x)
#         return self.fc(self.dropout(x.view(x.size(0), -1)))

# # ================================
# # TRAIN & COMPARE
# # ================================

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# num_classes = len(dataset.classes)

# scores = {}
# best_f1 = 0
# best_model = None
# best_name = ""

# for act in ["relu", "gelu", "leakyrelu"]:
#     simple = SimpleCNN(num_classes, act).to(device)
#     simple.train_model(train_loader, val_loader, 10, device, f"SimpleCNN-{act}")
#     sp, sl = simple.evaluate(val_loader, device)
#     sf1 = f1_score(sl, sp, average="macro")

#     deep = DeepCNN(num_classes, act).to(device)
#     deep.train_model(train_loader, val_loader, 10, device, f"DeepCNN-{act}")
#     dp, dl = deep.evaluate(val_loader, device)
#     df1 = f1_score(dl, dp, average="macro")

#     scores[f"Simple-{act}"] = sf1
#     scores[f"Deep-{act}"] = df1

#     if sf1 > best_f1:
#         best_f1, best_model, best_name = sf1, simple, f"Simple-{act}"
#     if df1 > best_f1:
#         best_f1, best_model, best_name = df1, deep, f"Deep-{act}"

# plt.figure(figsize=(8,5))
# plt.bar(scores.keys(), scores.values())
# plt.xticks(rotation=45)
# plt.ylabel("Macro F1 Score")
# plt.title("Activation Function Comparison")
# plt.show()


# bp, bl = best_model.evaluate(val_loader, device)
# cm = confusion_matrix(bl, bp)

# plt.figure(figsize=(8,6))
# sns.heatmap(cm, annot=True, fmt="d",
#             xticklabels=dataset.classes,
#             yticklabels=dataset.classes,
#             cmap="Blues")
# plt.title(f"Confusion Matrix – {best_name}")
# plt.show()

# torch.save(best_model.state_dict(), "best_activation_model.pth")

# print(f"\n BEST MODEL: {best_name}")
# print(f" BEST F1 SCORE: {best_f1:.4f}")




# import os
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import Dataset, DataLoader, random_split
# from PIL import Image
# import pandas as pd
# import torchvision.transforms as T
# import numpy as np
# import matplotlib.pyplot as plt
# from sklearn.metrics import confusion_matrix, f1_score
# import seaborn as sns
# from tqdm import tqdm


# class ButterflyDataset(Dataset):
#     def __init__(self, csv_file, img_folder, transform=None):
#         self.data = pd.read_csv(csv_file)
#         self.img_folder = img_folder
#         self.transform = transform
#         self.classes = sorted(self.data['label'].unique())
#         self.class_to_idx = {label: idx for idx, label in enumerate(self.classes)}

#     def __len__(self):
#         return len(self.data)

#     def __getitem__(self, idx):
#         filename = str(self.data.iloc[idx, 0])
#         img_file = [f for f in os.listdir(self.img_folder) if f.startswith(filename)][0]
#         image = Image.open(os.path.join(self.img_folder, img_file)).convert("RGB")
#         label = self.class_to_idx[self.data.iloc[idx, 1]]
#         if self.transform:
#             image = self.transform(image)
#         return image, label


# transform = T.Compose([
#     T.Resize((96, 96)),
#     T.ToTensor(),
#     T.Normalize([0.5]*3, [0.5]*3)
# ])

# train_csv = "/content/drive/MyDrive/Butterfly_Google_Colab/unique_images_with_labels.csv"
# train_folder = "unique_photos/unique_photos"

# dataset = ButterflyDataset(train_csv, train_folder, transform)

# val_size = int(0.2 * len(dataset))
# train_ds, val_ds = random_split(dataset, [len(dataset)-val_size, val_size])

# train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
# val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)


# def get_activation(name):
#     if name == "relu":
#         return nn.ReLU()
#     if name == "gelu":
#         return nn.GELU()
#     if name == "leakyrelu":
#         return nn.LeakyReLU(0.1)
#     raise ValueError("Invalid activation")


# class SimpleCNN(nn.Module):
#     def __init__(self, num_classes, activation):
#         super().__init__()
#         act = get_activation(activation)

#         self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
#         self.bn1 = nn.BatchNorm2d(32)

#         self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
#         self.bn2 = nn.BatchNorm2d(64)

#         self.conv3 = nn.Conv2d(64, 128, 3, padding=1)

#         self.pool = nn.MaxPool2d(2)
#         self.act = act
#         self.dropout = nn.Dropout(0.3)

#         self.fc = nn.Linear(128 * 12 * 12, num_classes)

#         self.criterion = nn.CrossEntropyLoss()
#         self.optimizer = optim.Adam(self.parameters(), lr=3e-4)

#     def forward(self, x):
#         x = self.pool(self.act(self.bn1(self.conv1(x))))
#         x = self.pool(self.act(self.bn2(self.conv2(x))))
#         x = self.pool(self.act(self.conv3(x)))
#         x = x.view(x.size(0), -1)
#         return self.fc(self.dropout(x))

#     def train_model(self, train_loader, val_loader, epochs, device, name):
#         print(f"\n Training {name}")
#         for epoch in range(epochs):
#             self.train()
#             running_loss = 0.0

#             pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
#             for x, y in pbar:
#                 x, y = x.to(device), y.to(device)

#                 self.optimizer.zero_grad()
#                 loss = self.criterion(self(x), y)
#                 loss.backward()
#                 self.optimizer.step()

#                 running_loss += loss.item()
#                 pbar.set_postfix(loss=loss.item())

#             avg_loss = running_loss / len(train_loader)
#             preds, labels = self.evaluate(val_loader, device)
#             f1 = f1_score(labels, preds, average="macro")

#             print(f"Epoch [{epoch+1}/{epochs}] | Loss: {avg_loss:.4f} | Val F1: {f1:.4f}")

#     def evaluate(self, loader, device):
#         self.eval()
#         preds, labels = [], []
#         with torch.no_grad():
#             for x, y in loader:
#                 out = self(x.to(device))
#                 preds.append(out.argmax(1).cpu())
#                 labels.append(y)
#         return torch.cat(preds), torch.cat(labels)


# class DeepCNN(SimpleCNN):
#     def __init__(self, num_classes, activation):
#         super().__init__(num_classes, activation)
#         act = get_activation(activation)

#         self.features = nn.Sequential(
#             nn.Conv2d(3, 32, 3, padding=1), act, nn.BatchNorm2d(32),
#             nn.Conv2d(32, 64, 3, padding=1), act, nn.BatchNorm2d(64),
#             nn.MaxPool2d(2),

#             nn.Conv2d(64, 128, 3, padding=1), act, nn.BatchNorm2d(128),
#             nn.Conv2d(128, 256, 3, padding=1), act, nn.BatchNorm2d(256),
#             nn.MaxPool2d(2),

#             nn.Conv2d(256, 256, 3, padding=1), act,
#             nn.MaxPool2d(2)
#         )

#         self.fc = nn.Linear(256 * 12 * 12, num_classes)
#         self.dropout = nn.Dropout(0.3)

#         self.criterion = nn.CrossEntropyLoss()
#         self.optimizer = optim.Adam(self.parameters(), lr=3e-4)

#     def forward(self, x):
#         x = self.features(x)
#         return self.fc(self.dropout(x.view(x.size(0), -1)))


# # ================================
# # STEP 1: Compare ReLU models
# # ================================
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# num_classes = len(dataset.classes)

# relu_scores = {}
# best_f1 = 0
# best_model = None
# best_name = ""

# # SimpleCNN with ReLU
# simple = SimpleCNN(num_classes, "relu").to(device)
# simple.train_model(train_loader, val_loader, 10, device, "SimpleCNN-RELU")
# sp, sl = simple.evaluate(val_loader, device)
# sf1 = f1_score(sl, sp, average="macro")
# relu_scores["SimpleCNN"] = sf1

# # DeepCNN with ReLU
# deep = DeepCNN(num_classes, "relu").to(device)
# deep.train_model(train_loader, val_loader, 10, device, "DeepCNN-RELU")
# dp, dl = deep.evaluate(val_loader, device)
# df1 = f1_score(dl, dp, average="macro")
# relu_scores["DeepCNN"] = df1

# if sf1 > df1:
#     best_model, best_name, best_f1 = simple, "SimpleCNN", sf1
# else:
#     best_model, best_name, best_f1 = deep, "DeepCNN", df1

# print(f"\nBest RE-LU model: {best_name} | F1: {best_f1:.4f}")

# # ================================
# # STEP 2: Run best model with GELU and LeakyReLU
# # ================================
# scores = {"ReLU": best_f1}

# for act in ["gelu", "leakyrelu"]:
#     if best_name == "SimpleCNN":
#         model = SimpleCNN(num_classes, act).to(device)
#     else:
#         model = DeepCNN(num_classes, act).to(device)

#     model.train_model(train_loader, val_loader, 10, device, f"{best_name}-{act.upper()}")
#     preds, labels = model.evaluate(val_loader, device)
#     f1 = f1_score(labels, preds, average="macro")
#     scores[act.upper()] = f1
#     print(f"{best_name} + {act.upper()} => F1: {f1:.4f}")

# # ================================
# # Visualize
# # ================================
# plt.figure(figsize=(6,5))
# plt.bar(scores.keys(), scores.values())
# plt.ylabel("Macro F1 Score")
# plt.title(f"Activation Comparison for {best_name}")
# plt.show()

# # Confusion matrix for the best activation
# best_act = max(scores, key=scores.get)
# if best_name == "SimpleCNN":
#     final_model = SimpleCNN(num_classes, best_act.lower()).to(device)
# else:
#     final_model = DeepCNN(num_classes, best_act.lower()).to(device)

# final_model.train_model(train_loader, val_loader, 10, device, f"{best_name}-{best_act}")
# preds, labels = final_model.evaluate(val_loader, device)
# cm = confusion_matrix(labels, preds)

# plt.figure(figsize=(8,6))
# sns.heatmap(cm, annot=True, fmt="d",
#             xticklabels=dataset.classes,
#             yticklabels=dataset.classes,
#             cmap="Blues")
# plt.title(f"Confusion Matrix – {best_name}-{best_act}")
# plt.show()

# torch.save(final_model.state_dict(), "best_activation_model.pth")
# print(f"\n BEST MODEL: {best_name}-{best_act}")
# print(f" BEST F1 SCORE: {scores[best_act]:.4f}")






import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import pandas as pd
import torchvision.transforms as T
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, f1_score
import seaborn as sns
from tqdm import tqdm
import cv2

# =========================
# Dataset
# =========================
class ButterflyDataset(Dataset):
    def __init__(self, csv_file, img_folder, transform=None):
        self.data = pd.read_csv(csv_file)
        self.img_folder = img_folder
        self.transform = transform
        self.classes = sorted(self.data['label'].unique())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        filename = str(self.data.iloc[idx, 0])
        img_file = [f for f in os.listdir(self.img_folder) if f.startswith(filename)][0]
        image = Image.open(os.path.join(self.img_folder, img_file)).convert("RGB")
        label = self.class_to_idx[self.data.iloc[idx, 1]]
        if self.transform:
            image = self.transform(image)
        return image, label

# =========================
# Transforms
# =========================
transform = T.Compose([
    T.Resize((96, 96)),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])

csv_path = "/content/drive/MyDrive/Butterfly_Google_Colab/unique_images_with_labels.csv"
img_path = "unique_photos/unique_photos"

dataset = ButterflyDataset(csv_path, img_path, transform)
val_size = int(0.2 * len(dataset))
train_ds, val_ds = random_split(dataset, [len(dataset)-val_size, val_size])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

# =========================
# Activation selector
# =========================
def get_activation(name):
    name = name.lower()
    if name == "relu":
        return nn.ReLU()
    if name == "gelu":
        return nn.GELU()
    if name == "leakyrelu":
        return nn.LeakyReLU(0.1)
    raise ValueError("Unknown activation")

# =========================
# SimpleCNN
# =========================
class SimpleCNN(nn.Module):
    def __init__(self, num_classes, activation):
        super().__init__()
        act = get_activation(activation)
        self.conv = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.BatchNorm2d(32), act, nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), act, nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), act, nn.MaxPool2d(2),
            nn.Conv2d(128,128,3,padding=1), nn.BatchNorm2d(128), act,
            nn.Conv2d(128,256,3,padding=1), nn.BatchNorm2d(256), act
        )
        self.fc = nn.Linear(256*12*12, num_classes)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

# =========================
# DeepCNN
# =========================
class DeepCNN(nn.Module):
    def __init__(self, num_classes, activation):
        super().__init__()
        act = get_activation(activation)
        self.features = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.BatchNorm2d(32), act,
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), act, nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), act, nn.MaxPool2d(2),
            nn.Conv2d(128,256,3,padding=1), nn.BatchNorm2d(256), act, nn.MaxPool2d(2),
            nn.Conv2d(256,256,3,padding=1), nn.BatchNorm2d(256), act
        )
        self.fc = nn.Linear(256*12*12, num_classes)
        self.dropout = nn.Dropout(0.4)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.fc(self.dropout(x))

# =========================
# Training function with tqdm
# =========================
def train_model(model, train_loader, val_loader, epochs, device):
    optimizer = optim.Adam(model.parameters(), lr=3e-4)
    criterion = nn.CrossEntropyLoss()

    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{epochs}]", leave=False)
        for x, y in loop:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            loop.set_postfix(train_loss=loss.item())

        train_loss = running_loss / len(train_loader)
        train_losses.append(train_loss)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                val_loss += criterion(model(x), y).item()

        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    return train_losses, val_losses

# =========================
# Train both CNNs + activations
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(dataset.classes)
activations = ["relu","gelu","leakyrelu"]
models_to_train = ["SimpleCNN","DeepCNN"]

all_results = {}

for net in models_to_train:
    all_results[net] = {}
    for act in activations:
        print(f"\n🚀 Training {net} with {act.upper()}")
        model = SimpleCNN(num_classes, act).to(device) if net=="SimpleCNN" else DeepCNN(num_classes, act).to(device)
        train_l, val_l = train_model(model, train_loader, val_loader, 10, device)
        # Evaluate F1
        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for x, y in val_loader:
                preds.append(model(x.to(device)).argmax(1).cpu())
                labels.append(y)
        preds = torch.cat(preds)
        labels = torch.cat(labels)
        f1 = f1_score(labels, preds, average="macro")

        all_results[net][act] = {
            "model": model,
            "train_loss": train_l,
            "val_loss": val_l,
            "f1": f1
        }

# =========================
# Plot Train/Val loss
# =========================
plt.figure(figsize=(8,5))
for net in models_to_train:
    for act in activations:
        plt.plot(all_results[net][act]["val_loss"], label=f"{net}-{act.upper()} Val", linestyle="--")
        plt.plot(all_results[net][act]["train_loss"], label=f"{net}-{act.upper()} Train")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train/Validation Loss Comparison")
plt.legend()
plt.grid(True)
plt.show()

# =========================
# F1 Score comparison plot
# =========================
plt.figure(figsize=(6,5))
f1_values = [all_results[net][act]["f1"] for net in models_to_train for act in activations]
labels = [f"{net}-{act.upper()}" for net in models_to_train for act in activations]
plt.bar(labels, f1_values)
plt.xticks(rotation=45)
plt.ylabel("Macro F1 Score")
plt.title("F1 Score Comparison: SimpleCNN vs DeepCNN")
plt.show()